### PydanticOutputParser

In [1]:
# 우선 LLM에 출력하게 할 재료 목록(ingredients)과 순서(steps)를 필드로 하는 Recipe 클래스를 Pydantic의 모델로 정의
from pydantic import BaseModel, Field

class Recipe(BaseModel):
    ingredients: list[str] = Field(description="ingredients of the dish")
    steps: list[str] = Field(description="steps to make the dish")

In [2]:
# 이 Recipe 클래스를 제공해 PydanticOutputParser를 생성
from langchain_core.output_parsers import PydanticOutputParser

output_parser = PydanticOutputParser(pydantic_object=Recipe)

In [3]:
# 그리고 PydanticOutputParser에서 프롬프트에 포함할 출력 형식 설명문을 작성
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"ingredients": {"description": "ingredients of the dish", "items": {"type": "string"}, "title": "Ingredients", "type": "array"}, "steps": {"description": "steps to make the dish", "items": {"type": "string"}, "title": "Steps", "type": "array"}}, "required": ["ingredients", "steps"]}
```


In [4]:
# 이어서 format_instructions를 사용한 ChatPromptTemplate을 생성
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "사용자가 입력한 요리의 레시피를 생각해 주세요.\n\n"
            "{format_instructions}"
        ),
        ("human", "{dish}")
    ]
)

prompt_with_format_instructions = prompt.partial(format_instructions=format_instructions)

In [5]:
# 이 ChatPromptTemplate에 예시로 입력을 제공해 보겠다.
prompt_value = prompt_with_format_instructions.invoke({"dish": "카레"})
print("=== role: system ===")
print(prompt_value.messages[0].content)
print("=== role: user ===")
print(prompt_value.messages[1].content)

=== role: system ===
사용자가 입력한 요리의 레시피를 생각해 주세요.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"ingredients": {"description": "ingredients of the dish", "items": {"type": "string"}, "title": "Ingredients", "type": "array"}, "steps": {"description": "steps to make the dish", "items": {"type": "string"}, "title": "Steps", "type": "array"}}, "required": ["ingredients", "steps"]}
```
=== role: user ===
카레


In [6]:
# 이 텍스트를 입력으로 LLM을 실행해 보겠다.
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv("../.env")

model = ChatOpenAI(model="gpt-6-luna")
ai_message = model.invoke(prompt_value)
print(ai_message.content)

{
  "ingredients": [
    "카레 가루 또는 고형 카레 100g",
    "돼지고기 또는 닭고기 200g",
    "감자 2개",
    "당근 1개",
    "양파 1개",
    "물 700ml",
    "식용유 1큰술",
    "밥 2~3공기"
  ],
  "steps": [
    "고기와 감자, 당근, 양파를 한입 크기로 썬다.",
    "냄비에 식용유를 두르고 양파를 볶다가 고기를 넣어 겉면이 익을 때까지 볶는다.",
    "감자와 당근을 넣고 2~3분 더 볶는다.",
    "물을 붓고 끓인 뒤, 재료가 익을 때까지 중약불에서 15~20분 끓인다.",
    "불을 약하게 줄이고 카레 가루나 고형 카레를 넣어 잘 풀어 준다.",
    "걸쭉해질 때까지 저어 가며 5분 정도 더 끓인 뒤 밥 위에 얹어 낸다."
  ]
}


In [7]:
# 이 응답을 Pydantic 모델의 인스턴스로 변환해서 사용하고 싶은 경우가 많다.
# 그 변환 처리도 PydanticOutputPParser를 사용하면 간단
recipe = output_parser.invoke(ai_message)
print(type(recipe))
print(recipe)

<class '__main__.Recipe'>
ingredients=['카레 가루 또는 고형 카레 100g', '돼지고기 또는 닭고기 200g', '감자 2개', '당근 1개', '양파 1개', '물 700ml', '식용유 1큰술', '밥 2~3공기'] steps=['고기와 감자, 당근, 양파를 한입 크기로 썬다.', '냄비에 식용유를 두르고 양파를 볶다가 고기를 넣어 겉면이 익을 때까지 볶는다.', '감자와 당근을 넣고 2~3분 더 볶는다.', '물을 붓고 끓인 뒤, 재료가 익을 때까지 중약불에서 15~20분 끓인다.', '불을 약하게 줄이고 카레 가루나 고형 카레를 넣어 잘 풀어 준다.', '걸쭉해질 때까지 저어 가며 5분 정도 더 끓인 뒤 밥 위에 얹어 낸다.']


### StrOutputParser

In [8]:
# 샘플 코드는 다음과 같다.
from langchain_core.messages import AIMessage
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

ai_message = AIMessage(content="안녕하세요. 저는 AI 어이스턴트입니다.")
ai_message = output_parser.invoke(ai_message)
print(type(ai_message))
print(ai_message)

<class 'langchain_core.messages.base.TextAccessor'>
안녕하세요. 저는 AI 어이스턴트입니다.
